# 00 — Environment setup (Colab + local)

**Day 1 task:** check that this project runs in Google Colab and locally.

Run the cells top to bottom. In Colab: `File > Save a copy in Drive` before editing.

Checks performed:
1. Runtime detection (Colab vs local)
2. Project files present (or cloned)
3. Dependencies installed
4. Core imports + versions
5. Data directories created
6. Outbound network access (needed for data downloads)

In [ ]:
import importlib
import os
import sys
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Running in Colab: {IN_COLAB}")
print(f"Python: {sys.version.split()[0]}")
print(f"Working directory: {Path.cwd()}")

In [ ]:
# --- Locate or fetch the project -------------------------------------
# Local: you already launched Jupyter from the repo root.
# Colab: clone the repository once (update the URL after pushing to GitHub).

REPO_URL = "https://github.com/mwitanyehita/global-power-outage-risk.git"

def find_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "requirements.txt").is_file() and (candidate / "src").is_dir():
            return candidate
    return None

root = find_root(Path.cwd())

if root is None and IN_COLAB:
    !git clone {REPO_URL}
    root = find_root(Path("/content"))
    if root is not None:
        os.chdir(root)

assert root is not None, "Project root not found - clone the repo or set GRIDRISK_ROOT."
os.environ["GRIDRISK_ROOT"] = str(root)
print(f"Project root: {root}")
print(f"Files: {sorted(p.name for p in root.iterdir())}")

In [ ]:
# --- Dependencies -----------------------------------------------------
# In Colab the preinstalled environment differs from the local venv,
# so install from requirements.txt every time (quiet, fast when cached).

if IN_COLAB:
    %pip install -q -r requirements.txt
else:
    print("Local environment: assuming requirements.txt is already installed.")

In [ ]:
# --- Core imports + versions -----------------------------------------

import pandas as pd
import numpy as np
import requests

import src.config as config
importlib.reload(config)

print(config.runtime_summary())
for module in (pd, np, requests):
    print(f"  {module.__name__:10s} {module.__version__}")

for optional in ("geopandas", "sklearn", "plotly", "folium"):
    try:
        mod = importlib.import_module(optional)
        print(f"  {optional:10s} {getattr(mod, '__version__', '?')}")
    except ImportError:
        print(f"  {optional:10s} MISSING - install requirements.txt")

In [ ]:
# --- Data directories -------------------------------------------------

config.ensure_dirs()
for d in config.DATA_DIRS:
    print(f"{d}  {'OK' if d.exists() else 'MISSING'}")

In [ ]:
# --- Network check ----------------------------------------------------
# Downloads (Days 2-5) need outbound HTTPS. Open-Meteo is used as a
# keyless, lightweight probe and also confirms a candidate weather source.

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": -1.29,
    "longitude": 36.82,
    "start_date": "2024-01-01",
    "end_date": "2024-01-02",
    "hourly": "temperature_2m",
}
try:
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    payload = r.json()
    n = len(payload.get("hourly", {}).get("time", []))
    print(f"Network OK - Open-Meteo returned {n} hourly rows for Nairobi")
except Exception as exc:  # noqa: BLE001
    print(f"Network check FAILED: {exc}")
    print("Data-download days will not work until this passes.")

## Setup checklist

| Check | Expected |
|---|---|
| Runtime detection | prints `True` in Colab, `False` locally |
| Project root | found, lists `src/`, `data/`, `notebooks/` |
| Dependencies | no `MISSING` lines |
| Data directories | all `OK` |
| Network | `Network OK - Open-Meteo returned 48 hourly rows` |

**Known caveats**

- Local venv runs Python 3.14; Colab currently runs 3.11/3.12. `requirements.txt`
  is intentionally unpinned at Day 1 — pin exact versions during Week 4 (Day 28).
- The GitHub clone URL above must be updated once the repository is pushed.
- Colab sessions are ephemeral: re-run this notebook at the start of every session,
  and keep raw downloads mirrored to Google Drive if bandwidth matters.